In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

nav = pd.read_csv("../data/processed/clean_nav.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

print(nav.shape)
print(performance.shape)
print(benchmark.shape)

In [ ]:
print(nav.columns.tolist())
print(benchmark.columns.tolist())

In [ ]:
#Task 1 - Daily returns
nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(
    ["amfi_code","date"]
)
nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)
nav.head()

annual_returns = []

for code, group in nav.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    n = len(returns)
    annual_return = (
        (1 + returns).prod()
    ) ** (252/n) - 1
    annual_returns.append(
        [code, annual_return]
    )

annual_returns = pd.DataFrame(
    annual_returns,
    columns=[
        "amfi_code",
        "annualized_return"
    ]
)
annual_returns.head()

In [ ]:
annual_returns.to_csv(
    "../data/processed/returns_computed.csv",
    index=False
)

In [ ]:
#Task 2 - CAGR
cagr_results = []

for code, group in nav.groupby("amfi_code"):

    group = group.sort_values("date")

    start_nav = group["nav"].iloc[0]
    end_nav = group["nav"].iloc[-1]

    years = (
        group["date"].max() -
        group["date"].min()
    ).days / 365

    cagr = (
        end_nav / start_nav
    ) ** (1/years) - 1

    cagr_results.append(
        [code,cagr]
    )

cagr_df = pd.DataFrame(
    cagr_results,
    columns=[
        "amfi_code",
        "cagr"
    ]
)

cagr_df.head()

In [ ]:
cagr_df.to_csv(
    "../data/processed/cagr_report.csv",
    index=False
)

In [ ]:
#Task 3 - Sharpe Ratio
rf = 0.065
sharpe_list = []

for code, group in nav.groupby("amfi_code"):

    returns = group["daily_return"].dropna()

    annual_return = returns.mean()*252

    annual_volatility = (
        returns.std()
        * np.sqrt(252)
    )

    sharpe = (
        annual_return - rf
    ) / annual_volatility

    sharpe_list.append(
        [code,sharpe]
    )

sharpe_df = pd.DataFrame(
    sharpe_list,
    columns=[
        "amfi_code",
        "sharpe_ratio"
    ]
)

sharpe_df.head()

In [ ]:
sharpe_df.to_csv(
    "../data/processed/sharpe_values.csv",
    index=False
)

In [ ]:
#Task 4 - Sortino Ratio
sortino_list = []

for code, group in nav.groupby("amfi_code"):

    returns = group["daily_return"].dropna()

    downside = returns[
        returns < 0
    ]

    annual_return = (
        returns.mean()*252
    )

    downside_std = (
        downside.std()
        * np.sqrt(252)
    )

    sortino = (
        annual_return - rf
    ) / downside_std

    sortino_list.append(
        [code,sortino]
    )

sortino_df = pd.DataFrame(
    sortino_list,
    columns=[
        "amfi_code",
        "sortino_ratio"
    ]
)

sortino_df.head()

In [ ]:
sortino_df.to_csv(
    "../data/processed/sortino_values.csv",
    index=False
)

In [ ]:
print(nav.columns.tolist())

print(benchmark.columns.tolist())

In [ ]:
from scipy.stats import linregress

benchmark["date"] = pd.to_datetime(benchmark["date"])

nifty100 = benchmark[
    benchmark["index_name"] == "NIFTY100"
].copy()

nifty100 = nifty100.sort_values("date")

nifty100["benchmark_return"] = (
    nifty100["close_value"].pct_change()
)

nifty100.head()

In [ ]:
alpha_beta = []

for code, group in nav.groupby("amfi_code"):

    merged = pd.merge(
        group,
        nifty100[["date","benchmark_return"]],
        on="date",
        how="inner"
    )

    merged = merged.dropna()

    if len(merged) > 30:

        slope, intercept, r_value, p_value, std_err = linregress(
            merged["benchmark_return"],
            merged["daily_return"]
        )

        beta = slope
        alpha = intercept * 252

        alpha_beta.append(
            [code, alpha, beta]
        )

alpha_beta_df = pd.DataFrame(
    alpha_beta,
    columns=[
        "amfi_code",
        "alpha",
        "beta"
    ]
)

alpha_beta_df.head()

In [ ]:
alpha_beta_df.to_csv(
    "../data/processed/alpha_beta.csv",
    index=False
)

print("alpha_beta.csv saved successfully")

In [ ]:
#Task 6 - Maximun Drawdown
max_dd_list = []

for code, group in nav.groupby("amfi_code"):

    group = group.sort_values("date")

    running_max = group["nav"].cummax()

    drawdown = (
        group["nav"] / running_max
    ) - 1

    max_dd = drawdown.min()

    max_dd_list.append([
        code,
        max_dd
    ])

max_dd_df = pd.DataFrame(
    max_dd_list,
    columns=[
        "amfi_code",
        "max_drawdown"
    ]
)

max_dd_df.head()

In [ ]:
max_dd_df.to_csv(
    "../data/processed/max_drawdown.csv",
    index=False
)

print("max_drawdown.csv saved successfully")

In [ ]:
#Task 7 - Fund Scorecard
scorecard = performance.copy()

scorecard = scorecard.merge(
    sharpe_df,
    on="amfi_code",
    how="left"
)

scorecard = scorecard.merge(
    alpha_beta_df,
    on="amfi_code",
    how="left"
)

scorecard = scorecard.merge(
    max_dd_df,
    on="amfi_code",
    how="left"
)

scorecard.head()

In [ ]:
scorecard["return_rank"] = scorecard[
    "return_3yr_pct"
].rank(ascending=False)

scorecard["sharpe_rank"] = scorecard[
    "sharpe_ratio_y"
].rank(ascending=False)

scorecard["alpha_rank"] = scorecard[
    "alpha_y"
].rank(ascending=False)

scorecard["expense_rank"] = scorecard[
    "expense_ratio_pct"
].rank(ascending=True)

scorecard["dd_rank"] = scorecard[
    "max_drawdown"
].rank(ascending=False)

In [ ]:
scorecard["fund_score"] = (
    scorecard["return_rank"] * 0.30 +
    scorecard["sharpe_rank"] * 0.25 +
    scorecard["alpha_rank"] * 0.20 +
    scorecard["expense_rank"] * 0.15 +
    scorecard["dd_rank"] * 0.10
)

scorecard["fund_score"] = (
    100 *
    (
        scorecard["fund_score"].max()
        -
        scorecard["fund_score"]
    )
    /
    (
        scorecard["fund_score"].max()
        -
        scorecard["fund_score"].min()
    )
)

scorecard = scorecard.sort_values(
    "fund_score",
    ascending=False
)

scorecard[
    [
        "amfi_code",
        "scheme_name",
        "fund_score"
    ]
].head(10)

In [ ]:
scorecard.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

print("Fund Scorecard Saved Successfully")

In [ ]:
#Task 8 - BenchMark Comparison chart
top5 = scorecard.head(5)["amfi_code"].tolist()

top5

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,8))

for code in top5:

    fund = nav[
        nav["amfi_code"] == code
    ]

    plt.plot(
        fund["date"],
        fund["nav"],
        label=str(code)
    )

plt.title(
    "Top 5 Funds NAV Comparison"
)

plt.xlabel("Date")
plt.ylabel("NAV")
plt.legend()

plt.show()

In [ ]:
tracking_errors = []

for code, group in nav.groupby("amfi_code"):

    merged = pd.merge(
        group,
        nifty100[["date", "benchmark_return"]],
        on="date",
        how="inner"
    )

    merged = merged.dropna()

    if len(merged) > 30:

        diff = (
            merged["daily_return"]
            -
            merged["benchmark_return"]
        )

        tracking_error = diff.std() * np.sqrt(252)

        tracking_errors.append([
            code,
            tracking_error
        ])

tracking_error_df = pd.DataFrame(
    tracking_errors,
    columns=[
        "amfi_code",
        "tracking_error"
    ]
)

tracking_error_df.head()

In [ ]:
tracking_error_df.to_csv(
    "../data/processed/tracking_error.csv",
    index=False
)

print("tracking_error.csv saved successfully")

In [ ]:
sip = pd.read_csv("dashboard/04_monthly_sip_inflows.csv")
print(sip.columns.tolist())